In [1]:
import sys

sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pydicom import dcmread
import os
import random
from natsort import natsorted, index_natsorted
import glob

from PIL import Image
import time
import copy
import tqdm
import re
import shutil
# import cv2

from sklearn.cluster import KMeans, DBSCAN

In [2]:
home_path = os.path.expanduser('~/')
images_path = f'/home/trevor/repos/ECHO-ML-View/data/first_transfer'

In [ ]:
subject_list = []
for subject in tqdm.tqdm(natsorted(glob.glob(images_path + '/*/*/'))):
    dicom_list = []
    for dicom in natsorted(glob.glob(subject + '/*.dcm')):
        dicom_list.append(dicom)
    subject_list.append(dicom_list)

In [ ]:
def derive_time(dicom):
    dcm = dcmread(dicom)
    if 'AcquisitionDateTime' in dcm: 
        hour = int(dcm.AcquisitionDateTime[8:10])
        minute = int(dcm.AcquisitionDateTime[10:12])
        second = int(dcm.AcquisitionDateTime[12:14])
    elif 'ContentTime' in dcm:
        hour = int(dcm.ContentTime[0:2])
        minute = int(dcm.ContentTime[2:4])
        second = int(dcm.ContentTime[4:6])
    return (hour*3600 + minute*60 + second)/3600

def get_subject_times(subject):
    acq_times = []
    for dicom in subject:
        try:
            acq_times.append((derive_time(dicom),dicom))
        except:
            print(f'failed on {dicom}')
            acq_times.append([0, subject])
    earliest = acq_times[0][0]
    acq_times = [(time[0]-earliest, time[1]) for time in acq_times]
    acq_times = [(time[0]+3600*24, time[1]) if time[0] < 0 else (time[0], time[1]) for time in acq_times]
    return acq_times

def prepost_op_kmeans(subject_times, inclusion_width = 0.5):
    times = np.array([time[0] for time in subject_times]).reshape(-1,1)
    kmeans = KMeans(n_clusters=2, random_state=0).fit(times)
    if kmeans.labels_[0] == 1:
        kmeans.labels_ = [-label+1 for label in kmeans.labels_]
        
    #centroid based outlier removal
    centroids = sorted(kmeans.cluster_centers_)
    intraop_zone = [((centroids[1]+centroids[0])/2)-((centroids[1]-centroids[0])*inclusion_width*0.5), ((centroids[1]+centroids[0])/2)+((centroids[1]-centroids[0])*inclusion_width*0.5)]
    for i,time in enumerate(times):
        if time > intraop_zone[0] and time < intraop_zone[1]:
            kmeans.labels_[i] = -1
            
    labeled = pd.DataFrame(list(zip([s[1] for s in subject_times], [s[0] for s in subject_times], kmeans.labels_)), columns = ['clip_path','time','prepost_pred'])
    return labeled

def process_and_save(subject, savepath='./predictions'):
    subject_id = os.path.split(os.path.split(subject[0])[0])[1]
    result = prepost_op_kmeans(get_subject_times(subject))
    result.to_csv(f'{savepath}/prepost_pred_{subject_id}.csv', header=True)

In [ ]:
import multiprocessing

with multiprocessing.Pool(24) as pool:
    pool.map(process_and_save, tqdm.tqdm(subject_list))

In [ ]:
# Plotting for checking
cols = 5
fig, ax = plt.subplots(28, cols, tight_layout=True, figsize=(20, 30))
for i, subject_csv in tqdm.tqdm(enumerate(glob.glob('predictions/*.csv')[::50])):
    subject = pd.read_csv(subject_csv)
    ax[int(i/cols)][i%cols].hist(list(subject.loc[subject['prepost_pred'] == 0]['time']), bins=np.linspace(0, 15, 60), label='pre op', color='b')
    ax[int(i/cols)][i%cols].hist(list(subject.loc[subject['prepost_pred'] == 1]['time']), bins=np.linspace(0, 15, 60), label='post op', color='g')
    ax[int(i/cols)][i%cols].hist(list(subject.loc[subject['prepost_pred'] ==-1]['time']), bins=np.linspace(0, 15, 60), label='intra op', color='r')
    ax[int(i/cols)][i%cols].set_title(f'S{i*50}')
# plt.show()
plt.savefig('time_clustering.png',format='png',dpi=300)

In [3]:
csvs = '/home/trevor/repos/ECHO-ML-View/prepostop/predictions'
csvs = glob.glob(csvs + '/*.csv')

In [4]:
df_list = []
for csv in tqdm.tqdm(csvs):
    df_list.append(pd.read_csv(csv))

100%|██████████| 6900/6900 [00:04<00:00, 1564.33it/s]


In [34]:
prepostop = pd.concat(df_list)
prepostop = prepostop.reset_index().drop(columns=['index', 'Unnamed: 0'])

pd.options.mode.chained_assignment = None

for i in tqdm.tqdm(range(len(prepostop))):
    prepostop['clip_path'][i] = prepostop['clip_path'][i][54:]

100%|██████████| 734195/734195 [01:12<00:00, 10097.74it/s]


In [35]:
prepostop

,clip_path,time,prepost_pred
0,48359237/1.2.840.113663.1500.1.458631247.3.2.2...,0.000000,0
1,48359237/1.2.840.113663.1500.1.458631247.3.3.2...,0.190000,0
2,48359237/1.2.840.113663.1500.1.458631247.3.4.2...,0.193056,0
3,48359237/1.2.840.113663.1500.1.458631247.3.5.2...,0.193889,0
4,48359237/1.2.840.113663.1500.1.458631247.3.6.2...,0.195278,0
...,...,...,...
734190,52360540/1.2.840.113663.1500.1.475091405.3.94....,5.644167,1
734191,52360540/1.2.840.113663.1500.1.475091405.3.95....,5.673889,1
734192,52360540/1.2.840.113663.1500.1.475091405.3.96....,5.686944,1
734193,52360540/1.2.840.113663.1500.1.475091405.3.97....,5.693611,1


In [36]:
prepostop.to_csv('../../labels/preop_postop_prediction.csv')

# Save images to tasks for CVAT labeling

In [ ]:
# Takes the already existing png files from a single folder and sorts them into individual folders based on subject and clip

directory = f'../data/multiview/train/me_4_ch'

subject_pattern = r'^([^_]+)'
clip_pattern = r'^[^_]+_([^_]+)'
frame_pattern = r'^[^_]+_[^_]+_([^\.]+)'

currsub = ''
currclip = ''

savedir = '../data/cvat'
if os.path.exists(savedir):
    shutil.rmtree(savedir)
os.makedirs(savedir)

for impath in tqdm(natsorted(glob.glob(f'{directory}/*.png'))):
    subject = re.match(subject_pattern, os.path.basename(impath)).group()
    clip = re.match(clip_pattern, os.path.basename(impath)).group()
    frame = re.match(frame_pattern, os.path.basename(impath)).group()
    if currsub != subject:
        currsub = subject
        os.makedirs(f'{savedir}/{currsub}')
    if currclip != clip:
        currclip = clip
        os.makedirs(f'{savedir}/{currsub}/{currclip}')
    savepath = f'{savedir}/{currsub}/{currclip}/{frame}.png'

    shutil.copyfile(impath, savepath)

In [ ]:
# Saves the video files for batch upload to CVAT

directory = '../data/cvat'

for subject in tqdm(natsorted(glob.glob(f'{directory}/*'))):
    for clip in natsorted(glob.glob(f'{subject}/*/')):
        images = [img for img in natsorted(glob.glob(f'{clip}*')) if img.endswith(".png")]
        frame = cv2.imread(images[0])
        height, width, layers = frame.shape

        video = cv2.VideoWriter(f'{clip[:-1]}.avi', cv2.VideoWriter_fourcc(*'XVID'), 30, (width,height))

        for image in images:
            video.write(cv2.imread(image))
        cv2.destroyAllWindows()
        video.release()